# Visualisation du Nettoyage des Données (Avant / Après)

Ce notebook utilise `pandas` pour illustrer concrètement l'impact de l'étape de nettoyage automatique sur nos événements OpenAgenda.
Nous allons comparer les données brutes (`events_raw.json`) avec les données propres prêtes pour le RAG (`events_clean.json`).

In [16]:
import pandas as pd
import json

# 1. Chargement des données brutes (Avant nettoyage)
with open('../data/events_raw.json', 'r', encoding='utf-8') as f:
    df_raw = pd.DataFrame(json.load(f))

# 2. Chargement des données propres (Après nettoyage)
with open('../data/events_clean.json', 'r', encoding='utf-8') as f:
    df_clean = pd.DataFrame(json.load(f))

pd.set_option('display.max_rows', None)

print("Données chargées avec succès pour la comparaison.")

Données chargées avec succès pour la comparaison.


In [17]:
df_raw.isnull().sum()

image                                                                                                                                             181
featured                                                                                                                                            0
attendanceMode                                                                                                                                      0
keywords                                                                                                                                            0
dateRange                                                                                                                                           0
timezone                                                                                                                                            0
imageCredits                                                                                        

In [18]:
seuil_tolerance = 0.50 # 50%
limite_valeurs_non_nulles = int((1 - seuil_tolerance) * len(df_raw))

# thresh = "Nombre minimum de valeurs valides requises pour garder la colonne"
df_raw = df_raw.dropna(axis=1, thresh=limite_valeurs_non_nulles)
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 6013 entries, 0 to 6012
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   image              5832 non-null   object
 1   featured           6013 non-null   bool  
 2   attendanceMode     6013 non-null   int64 
 3   keywords           6013 non-null   object
 4   dateRange          6013 non-null   object
 5   timezone           6013 non-null   str   
 6   imageCredits       3286 non-null   str   
 7   originAgenda       6013 non-null   object
 8   description        6013 non-null   object
 9   title              6013 non-null   object
 10  uid                6013 non-null   int64 
 11  lastTiming         6013 non-null   object
 12  firstTiming        6013 non-null   object
 13  location           5984 non-null   object
 14  slug               6013 non-null   str   
 15  status             6013 non-null   int64 
 16  parent_agenda_uid  6013 non-null   int64 
dtypes: boo

In [4]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 5984 entries, 0 to 5983
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   uid             5984 non-null   int64 
 1   slug            5984 non-null   str   
 2   title_fr        5984 non-null   str   
 3   description_fr  5984 non-null   str   
 4   keywords_fr     5984 non-null   object
 5   location        5984 non-null   object
 6   firstTiming     5984 non-null   object
 7   lastTiming      5984 non-null   object
 8   attendanceMode  5984 non-null   int64 
dtypes: int64(2), object(4), str(3)
memory usage: 420.9+ KB


## 1. Bilan Général : Doublons et Valeurs Manquantes
Le tableau suivant compare le volume de données, la présence de doublons (basé sur l'identifiant `uid`) et le nombre d'événements sans adresse.

In [19]:
# Calcul des statistiques comparatives
stats_data = {
    "Étape": ["Avant Nettoyage (Brut)", "Après Nettoyage (Propre)"],
    "Nombre total d'événements": [len(df_raw), len(df_clean)],
    "Doublons (UID)": [df_raw.duplicated(subset=['uid']).sum(), df_clean.duplicated(subset=['uid']).sum()],
    "Sans Localisation (Valeurs manquantes)": [df_raw['location'].isnull().sum(), df_clean['location'].isnull().sum()]
}

df_stats = pd.DataFrame(stats_data).set_index("Étape")

# Affichage visuel sous forme de tableau Pandas
df_stats

,Nombre total d'événements,Doublons (UID),Sans Localisation (Valeurs manquantes)
Étape,,,
Avant Nettoyage (Brut),6013,321,29
Après Nettoyage (Propre),5984,315,0


## 2. Valeurs incorrectes ou bruitées : Le problème du multilingue
Dans les données brutes, les champs textuels (titre, description) sont des dictionnaires contenant plusieurs langues (français, anglais, espagnol...). 
Cela constitue des **données incorrectes** ou bruitées pour notre RAG qui ne fonctionne qu'en français. 

Voici la comparaison visuelle d'un champ avant et après extraction de la version française :

In [20]:
print("\n❌ AVANT NETTOYAGE (Champ 'title' complet avec toutes les langues) :")
print(json.dumps(df_raw['title'].iloc[0], indent=2, ensure_ascii=False))

print("\n" + "-"*50)

print("\n✅ APRÈS NETTOYAGE (Champ 'title_fr' ciblé) :")
print(df_clean['title_fr'].iloc[0])


❌ AVANT NETTOYAGE (Champ 'title' complet avec toutes les langues) :
{
  "fr": "Journée Technique Hydrologie regénérative en forte pente"
}

--------------------------------------------------

✅ APRÈS NETTOYAGE (Champ 'title_fr' ciblé) :
Journée Technique Hydrologie regénérative en forte pente


## 3. Séparation du "Signal" sémantique et du "Bruit" technique
Même après avoir retiré les colonnes très vides, il nous restait des colonnes bien remplies mais inutiles pour le RAG (ex: `slug`, `timezone`, `imageCredits`).

**Pourquoi les supprimer ?** Pour optimiser les vecteurs mathématiques (Embeddings) de FAISS. On veut concentrer le modèle uniquement sur le sens du texte, sans polluer le calcul de similarité avec des mots-clés administratifs.

In [21]:
# Comparaison des colonnes (Signal vs Bruit)
colonnes_techniques_supprimees = ['slug', 'timezone', 'updatedAt', 'originAgenda', 'imageCredits']
colonnes_semantiques_conservees = ['title_fr', 'description_fr', 'location', 'keywords_fr', 'image', 'firstTiming', 'lastTiming']

df_explication = pd.DataFrame({
    "Type de Donnée": ["Bruit Technique (Supprimé)", "Signal Sémantique (Conservé)"],
    "Exemples de colonnes": [
        ", ".join(colonnes_techniques_supprimees),
        ", ".join(colonnes_semantiques_conservees)
    ],
    "Impact sur le RAG": [
        "Dilue la similarité vectorielle (FAISS) et consomme des tokens LLM inutilement.",
        "Concentre la recherche sur le vrai sens de l'événement."
    ]
})

df_explication.set_index("Type de Donnée")

,Exemples de colonnes,Impact sur le RAG
Type de Donnée,,
Bruit Technique (Supprimé),"slug, image, timezone, updatedAt, originAgenda...",Dilue la similarité vectorielle (FAISS) et con...
Signal Sémantique (Conservé),"title_fr, description_fr, location, keywords_fr",Concentre la recherche sur le vrai sens de l'é...


## 4. Aperçu visuel des données prêtes pour l'indexation FAISS
Voici à quoi ressemblent les premières lignes de notre DataFrame final, propre, unifié et prêt à être injecté dans notre base vectorielle.

In [22]:
cols_to_display = ['uid', 'title_fr', 'description_fr', 'location', 'firstTiming', 'lastTiming', 'image']
df_clean[cols_to_display].head()

,uid,title_fr,description_fr,location
0,95768697,Journée Technique Hydrologie regénérative en f...,demi-journée technique gratuite et ouverte à t...,"{'address': '2, place de la salle des fêtes 07..."
1,89578114,Webinaire – La réforme de la facturation élect...,La CA07 organise un webinaire pour comprendre ...,"{'address': '07000', 'city': 'Veyras', 'latitu..."
2,65182490,Webinaire Castanea Sativa,La Chambre d'agriculture de l'Ardèch propose u...,"{'address': '07000', 'city': 'Veyras', 'latitu..."
3,36521668,Estive en fête !,Randonnée pastorale festive sur les hauteurs d...,"{'address': 'la croix de Bauzon, Ardèche', 'ci..."
4,36284787,Facturation électronique,Réunion d’information interconsulaire sur la m...,"{'address': 'coucouron', 'city': 'Coucouron', ..."
